In [ ]:
import numpy as np
import pandas as pd

What code sections are most frequent?
Which ones most often lead to convictions?
Which ones have the most severe racial disparities?
In what localities (fips) are these disparities most severe?

In [2]:
cases = pd.read_csv('data100k.csv')
cases.head(3).T

,0,1,2
person_id,102090000000110,343221000000125,343221000000125
HearingDate,2019-02-28,2009-12-07,2011-01-20
CodeSection,A.46.2-862,B.46.2-301,A.46.2-707
codesection,covered elsewhere,covered elsewhere,covered elsewhere
ChargeType,Misdemeanor,Misdemeanor,Misdemeanor
chargetype,Misdemeanor,Misdemeanor,Misdemeanor
Class,1,1,3
DispositionCode,Guilty,Guilty,Guilty
disposition,Conviction,Conviction,Conviction
Plea,NaN,NaN,NaN


In [3]:
cases['CodeSection'].value_counts()

CodeSection
A.46.2-862    26379
B.46.2-301    25967
46.2-300      17934
C.46.2-862    11728
18.2-250.1    10573
              ...  
42-51             1
14.26             1
26-51(11)         1
A46.2-870         1
18.2-95/26        1
Name: count, Length: 4207, dtype: int64

In [4]:
cases['DispositionCode'].value_counts()

DispositionCode
Guilty                     156563
Nolle Prosequi              54680
Dismissed                   42520
Guilty In Absentia          31958
Not Guilty                   5807
Not Guilty/Acquitted         1623
Not True Bill                 250
No Indictment Presented       178
Dismissed/Other                19
Name: count, dtype: int64

In [6]:
cases['conviction'] = [x in ['Guilty', 'Guilty In Absentia'] for x in cases['DispositionCode']]

In [7]:
cases.head(3).T

,0,1,2
person_id,102090000000110,343221000000125,343221000000125
HearingDate,2019-02-28,2009-12-07,2011-01-20
CodeSection,A.46.2-862,B.46.2-301,A.46.2-707
codesection,covered elsewhere,covered elsewhere,covered elsewhere
ChargeType,Misdemeanor,Misdemeanor,Misdemeanor
chargetype,Misdemeanor,Misdemeanor,Misdemeanor
Class,1,1,3
DispositionCode,Guilty,Guilty,Guilty
disposition,Conviction,Conviction,Conviction
Plea,NaN,NaN,NaN


In [16]:
convict_rate = cases.groupby('CodeSection').agg({'conviction': ['count', 'mean']})
convict_rate = convict_rate.reset_index()
convict_rate.columns = ['CodeSection', 'count', 'mean']
convict_rate = convict_rate.query("count > 50")
convict_rate = convict_rate.sort_values('mean', ascending=False)

convict_rate

,CodeSection,count,mean
1806,23-55,55,0.981818
1755,23-22.1(A),131,0.954198
2103,29-17(C),70,0.942857
3961,A.46.2-862,26379,0.929414
3992,B.18.2-266,1879,0.905269
...,...,...,...
1436,19.2-123,125,0.096000
1102,18.2-374.1:1(A),112,0.089286
1450,19.2-135,83,0.084337
1479,19.2-99,339,0.000000


In [18]:
cases['Race'].unique()

array(['Black(Non-Hispanic)', 'Hispanic', 'White Caucasian(Non-Hispanic)',
       'MISSING', 'Asian Or Pacific Islander', 'Black (Non-Hispanic)',
       'White Caucasian (Non-Hispanic)',
       'Other(Includes Not Applicable.. Unknown)',
       'Other (Includes Not Applicable.. Unknown)', 'Black', 'White',
       'Unknown (Includes Not Applicable.. Unknown)', 'American Indian',
       'Unknown', 'Asian or Pacific Islander',
       'American Indian Or Alaskan Native'], dtype=object)

In [ ]:
replace_map = {'Black(Non-Hispanic)':'Black (Non-Hispanic)', 
               'Hispanic':'Hispanic', 
               'White Caucasian(Non-Hispanic)':'White (Non-Hispanic)',
               'MISSING':'Other or Missing', 
               'Asian Or Pacific Islander':'Asian or Pacific Islander', 
               'Black (Non-Hispanic)':'Black (Non-Hispanic)',
               'White Caucasian (Non-Hispanic)':'White (Non-Hispanic)',
               'Other(Includes Not Applicable.. Unknown)':'Other or Missing',
               'Other (Includes Not Applicable.. Unknown)':'Other or Missing',
               'Black':'Black (Non-Hispanic)',
               'White':'White (Non-Hispanic)',
               'Unknown (Includes Not Applicable.. Unknown)':'Other or Missing',
               'American Indian':'American Indian or Alaskan Native',
               'Unknown':'Other or Missing',
               'Asian or Pacific Islander':'Asian or Pacific Islander',
               'American Indian Or Alaskan Native':'American Indian or Alaskan Native'}
cases['Race'] = cases['Race'].replace(replace_map)
cases['Race'].unique()

Race
White (Non-Hispanic)                 159627
Black (Non-Hispanic)                 115627
Hispanic                               9319
Other or Missing                       5928
Asian or Pacific Islander              2794
American Indian or Alaskan Native       303
Name: count, dtype: int64

In [23]:
cases['Race'].value_counts()

Race
White (Non-Hispanic)                 159627
Black (Non-Hispanic)                 115627
Hispanic                               9319
Other or Missing                       5928
Asian or Pacific Islander              2794
American Indian or Alaskan Native       303
Name: count, dtype: int64

In [34]:
convict_rate_race = cases.groupby('Race').agg({'conviction': ['count', 'mean']})
convict_rate_race = convict_rate_race.reset_index()
convict_rate_race.columns = ['Race', 'count', 'mean']
convict_rate_race = convict_rate_race.sort_values('mean', ascending=False)

convict_rate_race

,Race,count,mean
3,Hispanic,9319,0.830347
0,American Indian or Alaskan Native,303,0.785479
4,Other or Missing,5928,0.739879
1,Asian or Pacific Islander,2794,0.662491
5,White (Non-Hispanic),159627,0.633715
2,Black (Non-Hispanic),115627,0.632638


In [47]:
convict_rate_race_cs = cases.groupby(['CodeSection', 'Race']).agg({'conviction': ['count', 'mean']})
convict_rate_race_cs = convict_rate_race_cs.reset_index()
convict_rate_race_cs.columns = ['CodeSection', 'Race', 'count', 'convictrate']
convict_rate_race_cs = convict_rate_race_cs.query("count > 30")

convict_rate_race_cs

,CodeSection,Race,count,convictrate
4,1-12,Black (Non-Hispanic),62,0.435484
75,10-42,White (Non-Hispanic),43,0.395349
76,10-43,Black (Non-Hispanic),41,0.170732
78,10-43,White (Non-Hispanic),82,0.353659
99,10-62,Black (Non-Hispanic),33,0.212121
...,...,...,...,...
6558,NO DMV,Black (Non-Hispanic),175,0.640000
6561,NO DMV,White (Non-Hispanic),202,0.608911
6620,Z.18.2-47,Black (Non-Hispanic),55,0.363636
6633,Z.18.2-91,Black (Non-Hispanic),131,0.725191


In [48]:
convict_rate_race_cs = convict_rate_race_cs.drop('count', axis=1)

convict_rate_race_cs

,CodeSection,Race,convictrate
4,1-12,Black (Non-Hispanic),0.435484
75,10-42,White (Non-Hispanic),0.395349
76,10-43,Black (Non-Hispanic),0.170732
78,10-43,White (Non-Hispanic),0.353659
99,10-62,Black (Non-Hispanic),0.212121
...,...,...,...
6558,NO DMV,Black (Non-Hispanic),0.640000
6561,NO DMV,White (Non-Hispanic),0.608911
6620,Z.18.2-47,Black (Non-Hispanic),0.363636
6633,Z.18.2-91,Black (Non-Hispanic),0.725191


In [50]:
convict_rate_wide = pd.pivot_table(convict_rate_race_cs, index = 'CodeSection', 
                                   columns = 'Race', 
                                   values = 'convictrate',)

convict_rate_wide

Race,American Indian or Alaskan Native,Asian or Pacific Islander,Black (Non-Hispanic),Hispanic,Other or Missing,White (Non-Hispanic)
CodeSection,,,,,,
1-12,NaN,NaN,0.435484,NaN,NaN,NaN
10-42,NaN,NaN,NaN,NaN,NaN,0.395349
10-43,NaN,NaN,0.170732,NaN,NaN,0.353659
10-62,NaN,NaN,0.212121,NaN,NaN,0.228261
13-1-5,NaN,NaN,0.578125,NaN,NaN,0.658537
...,...,...,...,...,...,...
G.46.2-870,NaN,NaN,NaN,NaN,NaN,0.593750
MISSING,NaN,NaN,0.589147,NaN,NaN,0.307692
NO DMV,NaN,NaN,0.640000,NaN,NaN,0.608911


In [53]:
convict_rate_wide['black_white_diff'] = convict_rate_wide['Black (Non-Hispanic)'] - convict_rate_wide['White (Non-Hispanic)']
convict_rate_wide = convict_rate_wide.sort_values('black_white_diff', ascending=False)

convict_rate_wide

Race,American Indian or Alaskan Native,Asian or Pacific Islander,Black (Non-Hispanic),Hispanic,Other or Missing,White (Non-Hispanic),black_white_diff
CodeSection,,,,,,,
MISSING,NaN,NaN,0.589147,NaN,NaN,0.307692,0.281455
23-10,NaN,NaN,0.448276,NaN,NaN,0.213592,0.234684
46.2-752,NaN,NaN,0.690647,NaN,NaN,0.492813,0.197834
19.2-128(B),NaN,NaN,0.660714,NaN,NaN,0.482143,0.178571
14.2-81,NaN,NaN,0.676190,NaN,NaN,0.500000,0.176190
...,...,...,...,...,...,...,...
D.18.2-266,NaN,NaN,NaN,NaN,NaN,0.763889,NaN
D.46.2-894,NaN,NaN,NaN,NaN,NaN,0.678571,NaN
G.18.2-266,NaN,NaN,NaN,NaN,NaN,0.909091,NaN
